# Pomysł na Architekturę Modelu

2-poziomowy stacking z naciskiem na tuning modeli, a nie ich ilość

- poziom 1: Algorytmy GBDT
    - Wielka trójca XGBoost, LightGBM i CatBoost
    - Do tego w miarę możliwości czasowych przynajmniej jeden model liniowy(LogisticRegression/RidgeClassifier),
    - przynajmniej jeden MLP, po tej bazie można rozbudowywać o SVM, więcej sieci, różne hiperparametry dla już wspomnianych modeli itd.
- poziom 2: Meta Learner
    - prawdopodobnie wystarczy coś prostego typu Logistic regression, ale można probować
    z np. mocno regularyzowanych LightGBM

---

# Foldowanie danych

- zwykły K-Fold
    - Absolutnie odpada. Target jest mocno niezbalansowanyc (zdecydowanie więcej osób spłaca kredyt niż tego nie robi), korzystając z K-Fold'a
    duża szana na otrzymania foldów typu: znaczna większość wartości targetu to 1

- Stratified K-Fold
    - Sensowny punkt wyjścia, w zasadzie to co K-Fold tylko z gwarancją utrzymania stałej proporcji klas targetu, a zatem eliminujący najważniejszy problem K-Folda

- Time Series Split
    - To by było ciekawe na jakiś prawdziwych danych, na syntetycznych raczej nie ma szans na coś takiego, ale jak już o tym czytałem to uznałem że wrzucę jako wiedza na przyszłość i potencjalnie ciekawostka do raportu na koniec. Trick polega na tym, żeby zobaczyć czy w jakieś cesze (u nas naturalne do tego byłoby id) nie kryje się jakiś drift czasowy. Czytaj w prawdziwym banku kolejni klienci mieliby z natury przypisywane coraz wyższe id, a co za tym idzie można by zaobserwować, że w kolejnych "koszykach" takich rekordów średnie wartości np. wysokości pożyczki czy średnie zarobki powinny być zbliżone do trendów z prawdziwego życia (Na przestrzeni ostatnich lat powinny rosnąć). Gdyby okazało się, że taka zależność istnieje to należałoby zrobić coś jak Stratified K-Fold, ale jeszcze z zaznaczeniem na grupy czasowe.
        - Poniżej kodzik do pokazania, że faktycznie nic takiego na danych syntetycznych nie występuje

- Group K-Fold
    - Służy do zapobiegania rozbijania na zbiory walidacyjny i treningowy grup, które z logicznych powodów nie powinne być rozbijane. Miałoby to dużo większe zastosowanie w naszym przykladzie jeżeli umielibyśmy pokazać, że np. konkretny klient wziął porzyczkę kilka razy. Wtedy słusznie jest kazać modelowi uczyć się/walidować klienta a nie jedno z jego działań.

In [1]:
import pandas as pd

df = pd.read_csv('train.csv')
df['id_bucket'] = pd.qcut(df['id'], q=10, labels=False)

drift_analysis = df.groupby('id_bucket')[
    ['loan_amount', 'annual_income', 'interest_rate', 'loan_paid_back']
].mean()

print(drift_analysis)

            loan_amount  annual_income  interest_rate  loan_paid_back
id_bucket                                                            
0          15025.112495   48216.556475      12.368354        0.796380
1          15037.936121   48209.472178      12.351021        0.797017
2          15010.284989   48114.165757      12.363412        0.799508
3          15060.391527   48371.673401      12.350306        0.798434
4          15047.209127   48174.317657      12.350887        0.800603
5          15022.188882   48044.515152      12.364109        0.799172
6          15004.881119   48132.210865      12.350956        0.799933
7          14999.641280   48343.651785      12.358161        0.798886
8          15034.788222   48239.498283      12.348345        0.799963
9          14960.543033   48275.965724      12.357901        0.798300


# Feature Engineering

- interakcje finansowe
    - Na ten moment mamy coś takiego jak debt_to_income, można by wyprodukować różne inne tego typu wskaźniki i zobaczyć jak wyglądają ich korelacje. Z rzeczy, na które wpadłem/wyczytałem
        - loan_to_income
        - estimated_annual_interest_cost : (loan_amount * (interest_rate / 100))
        - default_risk : Expected Loss = Probability of Default × Loss Given Default × Exposure at Default

- encoding
    - Target Encoding na zmiennych kategorycznych o potencjalnie wysokiej kardynalności (grade_subgrade, loan_purpose, employment_status)
        - Gówno polega na tym, że zastępujemy każdą kategorię średnim prawdopodobieństwem spłaty kredytu dla tej grupy i robi się elegancka hierarchia.
        - Potencjalnie lepsze niż stare dobre OHE, bo nie generujemy nowych wymiarów, więc nie dokładamy czasu treningu, z jedną uwagą
                - Gemini mówi coś takiego -- Błąd do uniknięcia: Target Encoding musi być liczony wyłącznie na foldzie treningowym i aplikowany na walidacyjnym wewnątrz pętli CV. Zrobienie tego na całym zbiorze przed podziałem na foldy to potężny wyciek danych (Data Leakage), który zrujnuje Twój lokalny wynik ROC AUC.

- Grupowanie
    - też myśle trochę zabawa na czutke podobnie jak przy pierwszym punkcie. Można porobić rzeczy pod tytułem 'income_to_education_median_ratio' itd. itp. i zobaczyć czy to poprawia cokolwiek, czy też nie


- Statystyczno-matematyczne szponty Mahoga
    - ogarnąłem na czym polega szpont, a więc tak:
        - model generatywny dodaje do oryginalnego zbioru danych jakiś szum (kredyt 12000 --> 12005.87)
        - różnica polega na tym 5.87, 1200 jest takie samo i na tej podstawie próbujesz dopasować
        - rozbijasz oryginalną cechę np. amount_loan na kolumny, w każdej kolumnie po kolei jedna cyfra: części setnych, części dziesiętnych, jedności itd
        - robisz KNN/K-Means czy cokolwiek podobnego, żeby zobaczyć, który rekord z oryginalnego zbioru może być potencjalnie tym, który został przerobiony
        - jeżeli dopasowanie się uda, to można dodać nową kolumnę do naszego zestawu original_target, która będzie po prostu przepisaną prawdziwą wartością targetu z oryginalnego zbioru danych (bo najprawdopodobniej target nie był w żaden sposób zaszumiony) i mamy czitersko mocną cechę
    - oczywiście szpont zakłada, że mamy dostęp do oryginalnego zbioru danych, ale z tego, co udało mi się ustalić, to jest dokładnie ten zbiór, co był na konkursie Kagglowym 5.11 (poniżej kodzik do sprawdzenia, że to te same sety). W takim razie wystarczyło dotrzeć tylko do informacji zamieszczonej na tronie konkursu, że było to trenowane na https://www.kaggle.com/datasets/nabihazahid/loan-prediction-dataset-2025/data.

In [2]:
kaggle_path = 'train_kaggle.csv'
dsc_path = 'train.csv'

def verify_dataset_match(local_data_path, competition_data_path):
    local_df = pd.read_csv(local_data_path)
    comp_df = pd.read_csv(competition_data_path)

    # 1. Shape check
    if local_df.shape != comp_df.shape:
        print(f"Shape mismatch: Local {local_df.shape} vs Competition {comp_df.shape}")
        return False

    # 2. Columns check
    local_columns = set(local_df.columns)
    comp_columns = set(comp_df.columns)

    if local_columns != comp_columns:
        print(f"Columns mismatch.")
        print(f"Only in local: {local_columns - comp_columns}")
        print(f"Only in competition: {comp_columns - local_columns}")
        return False

    # 3. Deep value comparison
    local_df = local_df.sort_values('id').reset_index(drop=True)
    comp_df = comp_df.sort_values('id').reset_index(drop=True)

    try:
        pd.testing.assert_frame_equal(local_df, comp_df)
        print("Perfect match! The datasets are exactly 1:1.")
        return True
    except AssertionError as error_msg:
        print("Datasets have the same shape and columns, but values or data types differ:")
        print(error_msg)
        return False

verify_dataset_match(dsc_path, kaggle_path)

Perfect match! The datasets are exactly 1:1.


True